In [1]:
#Loads the NuScenes dataset for further processing.
from nuscenes.nuscenes import NuScenes
from sklearn.model_selection import train_test_split
import numpy as np
import json
import os
import pandas as pd
import random
from tqdm import tqdm

# Load the full trainval set
nusc = NuScenes(version='v1.0-trainval', dataroot='/data/sets/nuscenes', verbose=True)


Loading NuScenes tables for version v1.0-trainval...
23 category,
8 attribute,
4 visibility,
64386 instance,
12 sensor,
10200 calibrated_sensor,
2631083 ego_pose,
68 log,
850 scene,
34149 sample,
2631083 sample_data,
1166187 sample_annotation,
4 map,
Done loading in 43.600 seconds.
Reverse indexing ...
Done reverse indexing in 7.4 seconds.


In [2]:
# Selects a subset of scenes and splits them into train/val/test for data preparation.
all_scenes = nusc.scene.copy()

selected_scenes = all_scenes[20:31]  # This gives 11 scenes: indices 20 to 30
from sklearn.model_selection import train_test_split

train_scenes, temp_scenes = train_test_split(selected_scenes, test_size=0.3, random_state=42)
val_scenes, test_scenes = train_test_split(temp_scenes, test_size=0.4, random_state=42)

In [3]:
#Defines a helper function to calculate object velocity using instance annotations.
def get_instance_velocity(nusc, instance_token, current_ann_token):
    try:
        instance = nusc.get('instance', instance_token)
        annotations = instance.get('anns', [])
        if current_ann_token not in annotations:
            return np.zeros(3)
        curr_idx = annotations.index(current_ann_token)
        if curr_idx == 0:
            return np.zeros(3)
        prev_ann_token = annotations[curr_idx - 1]
        curr_ann = nusc.get('sample_annotation', current_ann_token)
        prev_ann = nusc.get('sample_annotation', prev_ann_token)
        t1 = np.array(prev_ann['translation'])
        t2 = np.array(curr_ann['translation'])
        dt = (curr_ann['timestamp'] - prev_ann['timestamp']) * 1e-6
        if dt == 0:
            return np.zeros(3)
        return (t2 - t1) / dt
    except:
        return np.zeros(3)


In [4]:
#Processes each scene to compute risk metrics (TTC, relative speed, ego speed) and labels frames with a risk level.
import os
import pandas as pd
from tqdm import tqdm
import numpy as np

def process_scenes(scene_list, split_name="train", save_dir="./nuscenes_output"):
    os.makedirs(save_dir, exist_ok=True)
    data_rows = []

    for scene in tqdm(scene_list, desc=f"Processing {split_name} scenes"):
        scene_token = scene['token']
        sample_token = scene['first_sample_token']
        
        while sample_token:
            sample = nusc.get('sample', sample_token)
            timestamp = sample['timestamp']
            lidar_token = sample['data']['LIDAR_TOP']
            lidar_data = nusc.get('sample_data', lidar_token)
            ego_pose = nusc.get('ego_pose', lidar_data['ego_pose_token'])
            ego_translation = np.array(ego_pose['translation'])

            # Calculate ego velocity
            ego_velocity = np.zeros(3)
            if lidar_data['prev']:
                prev_lidar_data = nusc.get('sample_data', lidar_data['prev'])
                prev_pose = nusc.get('ego_pose', prev_lidar_data['ego_pose_token'])
                dt = (lidar_data['timestamp'] - prev_lidar_data['timestamp']) * 1e-6
                if dt > 0:
                    ego_velocity = (np.array(ego_pose['translation']) - np.array(prev_pose['translation'])) / dt
            ego_speed = np.linalg.norm(ego_velocity)

            # Find moving objects
            moving_objects = []
            for ann_token in sample['anns']:
                ann = nusc.get('sample_annotation', ann_token)
                instance_token = ann['instance_token']
                object_translation = np.array(ann['translation'])
                distance = np.linalg.norm(object_translation - ego_translation)
                object_velocity = get_instance_velocity(nusc, instance_token, ann_token)
                rel_velocity = np.linalg.norm(object_velocity - ego_velocity)
                ttc = distance / rel_velocity if rel_velocity > 0 else float('inf')

                if rel_velocity > 0.5:  # Filter moving objects
                    category = ann['category_name']
                    attribute = nusc.get('attribute', ann['attribute_tokens'][0])['name'] if ann['attribute_tokens'] else 'None'
                    moving_objects.append({
                        'category': category,
                        'distance': distance,
                        'rel_velocity': rel_velocity,
                        'ttc': ttc,
                        'attribute': attribute
                    })

            moving_objects = sorted(moving_objects, key=lambda x: x['distance'])[:2]

            # Pad if fewer than 2
            while len(moving_objects) < 2:
                moving_objects.append({
                    'category': 'None',
                    'distance': 0.0,
                    'rel_velocity': 0.0,
                    'ttc': float('inf'),
                    'attribute': 'None'
                })

            # Assign risk level
            risk_level = 'low'
            closest_ttc = moving_objects[0]['ttc']
            if closest_ttc < 1.5:
                risk_level = 'high'
            elif 1.5 <= closest_ttc <= 3.0:
                risk_level = 'medium'

            data_rows.append({
            'timestamp': timestamp,
            'scene_token': scene_token,
            'sample_token': sample_token,
            'ego_speed': round(ego_speed, 2),
            'object_1_category': moving_objects[0]['category'],
            'object_1_ttc': round(moving_objects[0]['ttc'], 2),
            'object_1_rel_velocity': round(moving_objects[0]['rel_velocity'], 2),
            'object_2_category': moving_objects[1]['category'],
            'object_2_ttc': round(moving_objects[1]['ttc'], 2),
            'object_2_rel_velocity': round(moving_objects[1]['rel_velocity'], 2),
            'risk_level': risk_level
            })

            sample_token = sample['next']
    
    df = pd.DataFrame(data_rows)
    csv_path = os.path.join(save_dir, f"nuscenes_risk_{split_name}.csv")
    df.to_csv(csv_path, index=False)
    print(f"✅ Saved {split_name} CSV to {csv_path}")
    return df
 

In [5]:
# Generates train/val/test CSV files and a merged dataset for model training and evaluation.
df_train = process_scenes(train_scenes, "train")
df_val = process_scenes(val_scenes, "val")
df_test = process_scenes(test_scenes, "test")

# Save DataFrames as CSV
df_train.to_csv("train.csv", index=False)
df_val.to_csv("val.csv", index=False)
df_test.to_csv("test.csv", index=False)

# Save combined flat file
df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)
df_all.to_csv("nuscenes_risk_all.csv", index=False)
print("✅ Saved: train.csv, val.csv, test.csv, nuscenes_risk_all.csv and their JSON sequences.")


Processing train scenes: 100%|███████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 91.56it/s]


✅ Saved train CSV to ./nuscenes_output\nuscenes_risk_train.csv


Processing val scenes: 100%|█████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 83.68it/s]


✅ Saved val CSV to ./nuscenes_output\nuscenes_risk_val.csv


Processing test scenes: 100%|████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 61.94it/s]

✅ Saved test CSV to ./nuscenes_output\nuscenes_risk_test.csv
✅ Saved: train.csv, val.csv, test.csv, nuscenes_risk_all.csv and their JSON sequences.


In [6]:
#Generates labeled time-series sequences from tabular data for training a prediction model.
def generate_sequences(df, output_json):
    sequence_length = 5
    sequence_inputs = []
    sequence_labels = []
    
    grouped = df.groupby('scene_token')
    for _, group in grouped:
        group = group.sort_values('timestamp').reset_index(drop=True)
        for i in range(len(group) - sequence_length):
            sequence = group.iloc[i:i + sequence_length]
            next_risk = group.iloc[i + sequence_length]['risk_level']
            ttc1_seq = sequence['object_1_ttc'].values
            ttc2_seq = sequence['object_2_ttc'].values
            ego_seq = sequence['ego_speed'].values

            input_seq = np.stack([ttc1_seq, ttc2_seq, ego_seq], axis=1)
            sequence_inputs.append(input_seq.tolist())
            sequence_labels.append(next_risk)
    
    with open(output_json, "w") as f:
        json.dump({"inputs": sequence_inputs, "labels": sequence_labels}, f)
    
    print(f"✅ Saved {len(sequence_inputs)} sequences to {output_json}")
 

In [7]:
# Creates JSON files with input sequences and labels for all dataset splits.
generate_sequences(df_train, "sequences_train.json")
generate_sequences(df_val, "sequences_val.json")
generate_sequences(df_test, "sequences_test.json")


✅ Saved 245 sequences to sequences_train.json
✅ Saved 69 sequences to sequences_val.json
✅ Saved 70 sequences to sequences_test.json


In [8]:
#Loads and encodes training data for model input.
import json
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load from JSON
with open("sequences_train.json", "r") as f:
    train_data = json.load(f)

X_train = np.array(train_data["inputs"])
y_train = np.array(train_data["labels"])

# Encode labels
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)

# Print shapes and class names
print(f"\n✅ X_train shape: {X_train.shape}")
print(f"✅ y_train shape: {y_train_enc.shape}")
print(f"✅ Label classes: {label_encoder.classes_}")



✅ X_train shape: (245, 5, 3)
✅ y_train shape: (245,)
✅ Label classes: ['high' 'low' 'medium']


In [9]:
#Prepares the full dataset with one-hot encoded labels for training a classification model.

import json
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

def load_sequences(path):
    with open(path, 'r') as f:
        data = json.load(f)
    X = np.array(data['inputs'])  # shape: (num_samples, 5, 3)
    y = np.array(data['labels'])  # shape: (num_samples,)
    return X, y

# Load training, validation, test data
X_train, y_train = load_sequences("sequences_train.json")
X_val, y_val = load_sequences("sequences_val.json")
X_test, y_test = load_sequences("sequences_test.json")

# Encode labels (low/medium/high → 0/1/2)
label_encoder = LabelEncoder()
y_train_enc = to_categorical(label_encoder.fit_transform(y_train))
y_val_enc = to_categorical(label_encoder.transform(y_val))
y_test_enc = to_categorical(label_encoder.transform(y_test))


In [10]:
#Defines the LSTM model architecture for predicting risk level from sequences.

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(5, 3), return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')  # 3 risk levels: low, medium, high
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


C:\Users\$8CUJ00-BGKRQC8B3RAL\AppData\Roaming\Python\Python39\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 64)                  │          17,408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │              99 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 19,587 (76.51 KB)

 Trainable params: 19,587 (76.51 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
#Trains the LSTM model for 30 epochs on the prepared sequences.
history = model.fit(
    X_train, y_train_enc,
    validation_data=(X_val, y_val_enc),
    epochs=30,
    batch_size=32
)


Epoch 1/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.3234 - loss: 1.0711 - val_accuracy: 0.1159 - val_loss: 1.0988
Epoch 2/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3762 - loss: 1.0978 - val_accuracy: 0.1159 - val_loss: 1.0992
Epoch 3/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4245 - loss: 1.0966 - val_accuracy: 0.1159 - val_loss: 1.0995
Epoch 4/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3953 - loss: 1.0967 - val_accuracy: 0.1159 - val_loss: 1.0997
Epoch 5/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3844 - loss: 1.0967 - val_accuracy: 0.1159 - val_loss: 1.0998
Epoch 6/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3889 - loss: 1.0950 - val_accuracy: 0.1159 - val_loss: 1.0997
Epoch 7/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4068 - loss: 1.0942 - val_accuracy: 0.1159 - val_loss: 1.0998
Epoch 8/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3948 - loss: 1.0930 - val_accuracy: 0.1159 - val_loss: 1.0999


In [12]:
#Tests the model’s accuracy on unseen data.
loss, accuracy = model.evaluate(X_test, y_test_enc)
print(f"Test Accuracy: {accuracy * 100:.2f}%")


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5847 - loss: 1.0638
Test Accuracy: 44.29%


In [13]:
#Persists the trained LSTM model to disk.
model.save("lstm_risk_model.keras")
print("✅ Model saved as lstm_risk_model.keras")

model.save("lstm_risk_model.h5")
print("✅ Model saved as lstm_risk_model.h5")


✅ Model saved as lstm_risk_model.keras
✅ Model saved as lstm_risk_model.h5


In [14]:
#Predicts the risk level at any frame using the 5-frame history as input.
def predict_risk_for_frame(frame_index, df_flat, model, label_encoder):
    """
    Predicts risk level at frame_index based on the 5 previous frames.
    
    Parameters:
        frame_index: int - the index (in df_flat) for which you want to predict risk level
        df_flat: pandas DataFrame - should include columns: 'object_1_ttc', 'object_2_ttc', 'ego_speed'
        model: trained LSTM model
        label_encoder: encoder for risk_level

    Returns:
        Predicted risk level and prints the 5-frame window used
    """
    if frame_index < 5:
        raise ValueError("Need at least 5 previous frames. Start from frame_index = 5 or higher.")

    # Extract the 5 previous rows
    window = df_flat.iloc[frame_index - 5:frame_index]
    print("🔍 5 Previous Frames Used for Prediction:")
    print(window[['timestamp', 'object_1_ttc', 'object_2_ttc', 'ego_speed', 'risk_level']])

    # Create input array
    sequence = window[['object_1_ttc', 'object_2_ttc', 'ego_speed']].values
    sequence_array = np.expand_dims(sequence, axis=0)  # shape: (1, 5, 3)

    # Predict
    pred = model.predict(sequence_array)
    pred_label = label_encoder.inverse_transform([np.argmax(pred)])
    print(f"\n🧠 Predicted Risk Level at Frame {frame_index}: {pred_label[0]}")
    return pred_label[0]

In [15]:
#Demonstrates how to perform inference on a single frame using the trained model.
import pandas as pd

# Load the test data used to generate X_test
df_test_flat = pd.read_csv("test.csv")

# Predict risk at a specific frame index (e.g., 200)
predicted = predict_risk_for_frame(20, df_test_flat, model, label_encoder)


🔍 5 Previous Frames Used for Prediction:
           timestamp  object_1_ttc  object_2_ttc  ego_speed risk_level
15  1531885984949356          8.36         10.18       1.22        low
16  1531885985449786         10.46         12.36       1.01        low
17  1531885985949658         11.90         13.69       0.91        low
18  1531885986449554         12.02         13.27       0.93        low
19  1531885986948891         10.50         11.09       1.05        low
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step

🧠 Predicted Risk Level at Frame 20: low
